In [16]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import ParameterGrid
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA
from sklearn.linear_model import ElasticNet
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor
from scipy.stats import t
import matplotlib.pyplot as plt
import matplotlib
from sklearn.pipeline import Pipeline

from statsmodels.stats.sandwich_covariance import cov_hac #heteroscedasticity and autocorrelation robust covariance matrix (Newey-West)
from statsmodels.regression.linear_model import OLS
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.stats.stattools import durbin_watson
import statsmodels.api as sm

In [17]:
df_final = pd.read_excel("data/processed/df_final.xlsx")
df_final = df_final.sort_values("Date").reset_index(drop=True) 

In [18]:
#check duplicate + columns
dup = df_final.duplicated(["Date","Ticker"]).sum()

print("Dupli init (Date,Ticker):", dup)
print(df_final.columns)

Dupli init (Date,Ticker): 0
Index(['Ticker', 'Date', 'dolvol', 'maxret', 'retvol', 'mom36m', 'mom12m',
       'mom6m', 'mom1m', 'chmom', 'turn', 'indmom', 'baspread', 'illiq',
       'stdturn', 'beta', 'beta_squared', 'idiovol', 'mvel1', 'agr', 'cashpr',
       'chinv', 'chsh', 'depr', 'dy', 'ep', 'invest', 'rd_mve', 'sp', 'nincr',
       'excess_return'],
      dtype='object')


In [19]:
# Avant de normaliser > on regarde la dispersion de nos variables → on le commente dans notre recherche
summary_stats = df_final.describe().T[["mean", "std"]]

latex_table = summary_stats.to_latex(index=True, float_format="%.4f")
print(latex_table)

\begin{tabular}{lrr}
\toprule
 & mean & std \\
\midrule
dolvol & 21.3847 & 1.6156 \\
maxret & 0.0392 & 0.0282 \\
retvol & 0.0184 & 0.0111 \\
mom36m & 0.5994 & 0.9311 \\
mom12m & 0.1756 & 0.3870 \\
mom6m & 0.0862 & 0.2404 \\
mom1m & 0.0144 & 0.0907 \\
chmom & 0.0026 & 0.3435 \\
turn & 0.0073 & 0.0073 \\
indmom & 0.1751 & 0.2662 \\
baspread & 0.0106 & 0.0225 \\
illiq & 0.0000 & 0.0000 \\
stdturn & 0.0031 & 0.0042 \\
beta & 1.0000 & 0.4363 \\
beta_squared & 1.1904 & 1.0814 \\
idiovol & 0.0333 & 0.0151 \\
mvel1 & 23.6895 & 1.4658 \\
agr & 0.0984 & 0.2297 \\
cashpr & 28.7304 & 368.9260 \\
chinv & 0.0032 & 0.0595 \\
chsh & 0.0072 & 0.0900 \\
depr & 0.0087 & 0.0055 \\
dy & -0.0017 & 0.0015 \\
ep & 0.0036 & 0.0093 \\
invest & 0.0387 & 0.1061 \\
rd_mve & 0.0034 & 0.0075 \\
sp & 0.0716 & 0.0916 \\
nincr & 0.4611 & 2.3890 \\
excess_return & 0.0123 & 0.0906 \\
\bottomrule
\end{tabular}



Composition du notebook : 
I. Traitement des données (time split, gestion des NaN) et fonctions des métriques 
II. Algorithmes
III. Résultats

I. Traitement des données (time split, gestion des NaN) et fonctions des métriques 

In [20]:
covariates = ["dolvol", "maxret", "retvol", "mom36m", "mom12m", "mom6m", "mom1m", "chmom", "turn", "indmom", "baspread", "illiq", "stdturn", "beta", "beta_squared",
"idiovol", "mvel1", "agr", "chinv", "cashpr", "chsh", "depr", "dy", "ep", "invest", "rd_mve", "sp", "nincr"] 

In [21]:
def generate_time_splits(df, date_col='Date',
                                   val_months=12,
                                   test_months=12,
                                   step_months=12,
                                   min_train_months=192):
    """
    Génère des splits temporels:
    - Train cumulatif (augmente d'un an à chaque refit)
    - Train 306 mois (= 85% de la data)
    - Validation = fenêtre fixe glissante de 1 an
    - Test = 1 an 
    - Avance de step_months à chaque itération : 1 an

    Paramètres :
    - df : DataFrame trié par date
    - date_col : nom de la colonne des dates
    - val_months : taille de la validation
    - test_months : taille du test 
    - step_months : pas de glissement
    - min_train_months : nombre minimum de mois de train initial

    Retour :
    - splits : liste de tuples (train_idx, val_idx, test_idx)
    """

    #On coupe chronologiquement donc on trie par date 
    df = df.sort_values(date_col).reset_index(drop=True)
    dates = sorted(df[date_col].unique())
    total_months = len(dates)

    splits = []

    #On démarre après avoir au moins min_train_months pour le train
    start = min_train_months
    while True:
        train_end = start  # train va de 0 jusqu'à train_end
        val_start = train_end
        val_end = val_start + val_months
        test_start = val_end
        test_end = test_start + test_months

        #Stop quand on a plus assez pour test
        if test_end > total_months:
            break

        train_dates = dates[:train_end]  
        val_dates = dates[val_start:val_end]
        test_dates = dates[test_start:test_end]

        train_idx = df[df[date_col].isin(train_dates)].index.tolist()
        val_idx = df[df[date_col].isin(val_dates)].index.tolist()
        test_idx = df[df[date_col].isin(test_dates)].index.tolist()

        splits.append((train_idx, val_idx, test_idx))

        #Décale fenêtre de un → on réactualise tous les 1 ans
        start += step_months

    return splits

In [22]:
def preprocess_split(x_train, x_val, x_test, covariates):
    """
    Impute les NaN par moyenne par Ticker (fit sur train),
    puis normalise chaque covariable entre -1 et 1 par date (rang cross-sectionnel).
    """
    means_by_ticker = x_train.groupby("Ticker")[covariates].mean(numeric_only=True)

    def fill_na_with_means(df):
        df = df.copy()
        for c in covariates:
            m = means_by_ticker[c].to_dict()
            df[c] = df[c].fillna(df["Ticker"].map(m))
        return df

    x_train_imp = fill_na_with_means(x_train)
    x_val_imp   = fill_na_with_means(x_val)
    x_test_imp  = fill_na_with_means(x_test)

    assert not x_train_imp[covariates].isna().any().any()
    assert not x_val_imp[covariates].isna().any().any()
    assert not x_test_imp[covariates].isna().any().any()
    
    #Normalisation cross-sectionnelle : par date
    def normalize_by_date(df):
        df = df.copy()
        out = pd.DataFrame(index=df.index, columns=covariates)
        for date_key, group_idx in df.groupby("Date").groups.items():
            sub = df.loc[group_idx, covariates]
            for cov in covariates:
                temp = sub[cov].dropna().sort_values()
                n = len(temp)
                if n == 1:
                    scores = pd.Series([0.0], index=temp.index)
                else:
                    scores = pd.Series(
                        2 * np.arange(n) / (n - 1) - 1, index=temp.index
                    )
                out.loc[temp.index, cov] = scores
        return out.astype(float)

    x_train_scaled = normalize_by_date(x_train_imp)
    x_val_scaled   = normalize_by_date(x_val_imp)
    x_test_scaled  = normalize_by_date(x_test_imp)

    #garde l'information de la date et du ticker
    x_train_scaled = pd.concat([x_train_imp[['Ticker','Date']].reset_index(drop=True), x_train_scaled.reset_index(drop=True)], axis=1)
    x_val_scaled   = pd.concat([x_val_imp[['Ticker','Date']].reset_index(drop=True), x_val_scaled.reset_index(drop=True)], axis=1)
    x_test_scaled  = pd.concat([x_test_imp[['Ticker','Date']].reset_index(drop=True), x_test_scaled.reset_index(drop=True)], axis=1)

    return x_train_scaled, x_val_scaled, x_test_scaled

In [23]:
def get_x_y(df, idx, target="excess_return"):
    subset = df.loc[idx].copy()
    x = subset.drop(columns=[target])
    y = subset[target]
    return x, y

def get_y_ha(df, idx, col="ha_global"):
    subset = df.loc[idx].copy()
    return subset[col]

In [24]:
#BENCHMARK HA : déjà créé avant on l'exporte et on le coupe "comme il faut"
df_ha = pd.read_excel("data/processed/df_ha.xlsx")
df_ha = df_ha[df_ha["Date"] >= "1990-12"].reset_index(drop=True)
df_ha = df_ha[df_ha["Date"] < "2020-12"].reset_index(drop=True)

In [25]:
#robustesse + check 

for name, df in [("df_final", df_final), ("df_ha", df_ha)]:
    print(f"\n--- {name} ---")
    print("Première date :", df["Date"].min())
    print("Dernière date :", df["Date"].max())
    print("Nombre de lignes :", len(df))
    print("Doublons (Date,Ticker) :", df.duplicated(["Date","Ticker"]).sum())


--- df_final ---
Première date : 1990-12
Dernière date : 2020-11
Nombre de lignes : 21240
Doublons (Date,Ticker) : 0

--- df_ha ---
Première date : 1990-12
Dernière date : 2020-11
Nombre de lignes : 21240
Doublons (Date,Ticker) : 0


In [26]:
#On découpe les splits puis on applique la gestion des NaN et la normalisation définie plus haut
splits = generate_time_splits(df_final)

preprocessed_splits = []

for train_idx, val_idx, test_idx in tqdm(splits):
    x_train, y_train = get_x_y(df_final, train_idx)
    x_val, y_val = get_x_y(df_final, val_idx)
    x_test, y_test = get_x_y(df_final, test_idx)

    #Imputation + Normalisation
    x_train, x_val, x_test = preprocess_split(x_train, x_val, x_test, covariates) #on enlève ticker et date

    preprocessed_splits.append((x_train, y_train, x_val, y_val, x_test, y_test))

100%|██████████| 13/13 [00:57<00:00,  4.40s/it]


In [27]:
"""Fonctions pour nos métriques : 
def r2: mesure le r2 selon la définition de Gu et al 
% ratio : success ratio, semblable à ce qui est fait dans le papier de Xiu et Liu
R2 benchmark : on compare le R2 de nos modèles à l'historical average 
"""

#r2
def r2(y_true, y_pred):
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum(y_true**2) 
    return 1 - ss_res/ss_tot if ss_tot != 0 else np.nan

#R2 benchmark 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)

#% ratio:
def success_ratio(y_true, y_pred, ignore_zero=True):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    sign_true = np.sign(y_true)
    sign_pred = np.sign(y_pred)

    if ignore_zero:
        mask = sign_true != 0
        sign_true = sign_true[mask]
        sign_pred = sign_pred[mask]

    if len(sign_true) == 0:
        return np.nan  
    return (sign_true == sign_pred).mean()

II. ALGORITHMES 
Pour chaque algo (HA, OLS, PLS, PCR, Enet, RF, GBRT, XGboost) on applique la même boucle d'entraînement, validation et test. Certains modèles étant particulièrement lents (RF et GBRT) on a décomposé le code en blocs distincts, un par modèle.  

Détails des listes: Chaque boucle produit les prédictions in-sample et out-of-sample, ainsi que les métriques associées. 

Listes communes à tous les modèles:
- y_true : vraies valeurs pour l'échantillon out-of-sample (test)
- y_trainval_true : vraies valeurs pour l’échantillon in-sample (train + val)
- dates_in, dates_oos: dates correspondantes aux observations in-sample et out-of-sample
- tickers_in, tickers_oos : pour l'instant, pas utilisé, pas utiles plus tard si on construit des portefeuilles

Ces listes sont initialisées une seule fois, dans le bloc du modèle de référence (Historical Average), et réutilisées dans tous les autres blocs.

Ensuite, pour chaque modèle, nous avons :
- y_trainval_pred_model : prédictions in-sample
- y_pred_model : prédictions oos
- r2_in_model : R² in sample, calculé par split 
- r2_oos_model : R² out-of-sample, calculé par split
- sucess_ratio_in_model et success_ratio_oos_model : utilisés pour calculer le success ratio 

Les résultats stockés dans ces listes sont ensuite utilisés dans la partie III. Résultats, pour l’analyse comparative des performances.

In [28]:
print(df_final.dtypes)

Ticker            object
Date              object
dolvol           float64
maxret           float64
retvol           float64
mom36m           float64
mom12m           float64
mom6m            float64
mom1m            float64
chmom            float64
turn             float64
indmom           float64
baspread         float64
illiq            float64
stdturn          float64
beta             float64
beta_squared     float64
idiovol          float64
mvel1            float64
agr              float64
cashpr           float64
chinv            float64
chsh             float64
depr             float64
dy               float64
ep               float64
invest           float64
rd_mve           float64
sp               float64
nincr            float64
excess_return    float64
dtype: object


In [29]:
#OLS

r2_in_ols, r2_oos_ols = [], []
success_ratio_in_ols, success_ratio_oos_ols = [], []
df_in_ols, df_oos_ols = [], []
feature_importance_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    y_trainval_pred = ols.predict(x_trainval)
    y_test_pred = ols.predict(x_test[covariates])

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_ols.append(r2_in)
    r2_oos_ols.append(r2_out)
    success_ratio_in_ols.append(sr_in)
    success_ratio_oos_ols.append(sr_out)

    feature_importance_ols.append(np.abs(ols.coef_))

    df_in_split = pd.DataFrame({
        "Split":  split_idx,
        "Date":   dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_ols": y_trainval_pred
    })
    df_in_ols.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split":  split_idx,
        "Date":   x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_ols": y_test_pred
    })
    df_oos_ols.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_ols = pd.concat(df_in_ols, ignore_index=True)
df_in_ols = df_in_ols.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")

df_oos_ols = pd.concat(df_oos_ols, ignore_index=True)

# Dates uniformisées 
df_in_ols["Date"]  = pd.to_datetime(df_in_ols["Date"]).dt.to_period("M")
df_oos_ols["Date"] = pd.to_datetime(df_oos_ols["Date"]).dt.to_period("M")

#Checks
print(f"Moyenne R² IN  : {np.mean(r2_in_ols):.6f}")
print(f"Moyenne R² OOS : {np.mean(r2_oos_ols):.6f}")

print("In sample lines:",  df_in_ols.shape[0],  "| Duplicates (Date,Ticker) =", df_in_ols.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_ols.shape[0], "| Duplicates (Date,Ticker) =", df_oos_ols.duplicated(["Date","Ticker"]).sum())

 46%|████▌     | 6/13 [00:00<00:00, 51.91it/s]

Split 1  R² IN: 0.028927 | OOS: -0.090321 | SR IN: 0.561 | SR OOS: 0.462
Split 2  R² IN: 0.020534 | OOS: 0.062708 | SR IN: 0.556 | SR OOS: 0.600
Split 3  R² IN: 0.024520 | OOS: 0.047838 | SR IN: 0.560 | SR OOS: 0.571
Split 4  R² IN: 0.025469 | OOS: -0.061079 | SR IN: 0.558 | SR OOS: 0.469
Split 5  R² IN: 0.023223 | OOS: 0.041146 | SR IN: 0.555 | SR OOS: 0.590
Split 6  R² IN: 0.023619 | OOS: 0.116290 | SR IN: 0.557 | SR OOS: 0.629
Split 7  R² IN: 0.025368 | OOS: 0.008153 | SR IN: 0.562 | SR OOS: 0.606
Split 8  R² IN: 0.025223 | OOS: -0.049367 | SR IN: 0.566 | SR OOS: 0.465
Split 9  R² IN: 0.023888 | OOS: 0.070714 | SR IN: 0.562 | SR OOS: 0.581


100%|██████████| 13/13 [00:00<00:00, 44.20it/s]

Split 10  R² IN: 0.024908 | OOS: 0.089959 | SR IN: 0.561 | SR OOS: 0.630
Split 11  R² IN: 0.025856 | OOS: -0.040769 | SR IN: 0.564 | SR OOS: 0.497
Split 12  R² IN: 0.024269 | OOS: 0.085293 | SR IN: 0.560 | SR OOS: 0.655
Split 13  R² IN: 0.025776 | OOS: 0.033992 | SR IN: 0.565 | SR OOS: 0.582
Moyenne R² IN  : 0.024737
Moyenne R² OOS : 0.024197
In sample lines: 20532 | Duplicates (Date,Ticker) = 0


Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [30]:
#PLS 
r2_in_pls, r2_oos_pls = [], []
success_ratio_in_pls, success_ratio_oos_pls = [], []
df_in_pls, df_oos_pls = [], []
best_components_pls, mse_val_grids = [], []

max_k = len(covariates)

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    # Sélection du meilleur k
    candidate_ks = range(1, max_k)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pls = PLSRegression(n_components=k, scale=False)
        pls.fit(x_train[covariates], y_train)
        y_val_pred = pls.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    best_components_pls.append(best_k)
    mse_val_grids.append(mse_val_grid)

    # Réentraînement final
    pls_final = PLSRegression(n_components=best_k, scale=False)
    pls_final.fit(x_trainval, y_trainval)

    y_trainval_pred = pls_final.predict(x_trainval).ravel()
    y_test_pred = pls_final.predict(x_test[covariates]).ravel()

    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pls.append(r2_in)
    r2_oos_pls.append(r2_out)
    success_ratio_in_pls.append(sr_in)
    success_ratio_oos_pls.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pls": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pls.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pls": y_test_pred
    })
    df_oos_pls.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pls = pd.concat(df_in_pls, ignore_index=True)
df_in_pls = df_in_pls.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")

df_oos_pls = pd.concat(df_oos_pls, ignore_index=True)

# Dates uniformisées 
df_in_pls["Date"]  = pd.to_datetime(df_in_pls["Date"]).dt.to_period("M")
df_oos_pls["Date"] = pd.to_datetime(df_oos_pls["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_pls.shape[0],  "| Duplicates (Date,Ticker) =", df_in_pls.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_pls.shape[0], "| Duplicates (Date,Ticker) =", df_oos_pls.duplicated(["Date","Ticker"]).sum())

  8%|▊         | 1/13 [00:01<00:19,  1.59s/it]

Split 1  R² IN: 0.023594 | OOS: -0.091327 | SR IN: 0.560 | SR OOS: 0.407


 15%|█▌        | 2/13 [00:03<00:17,  1.63s/it]

Split 2  R² IN: 0.019459 | OOS: 0.056390 | SR IN: 0.554 | SR OOS: 0.647


 23%|██▎       | 3/13 [00:05<00:17,  1.72s/it]

...

Split 11  R² IN: 0.022295 | OOS: -0.040104 | SR IN: 0.566 | SR OOS: 0.486


 92%|█████████▏| 12/13 [00:24<00:02,  2.39s/it]

Split 12  R² IN: 0.023626 | OOS: 0.088176 | SR IN: 0.562 | SR OOS: 0.664


100%|██████████| 13/13 [00:27<00:00,  2.10s/it]

Split 13  R² IN: 0.022472 | OOS: 0.031274 | SR IN: 0.567 | SR OOS: 0.578
In sample lines: 20532 | Duplicates (Date,Ticker) = 0
Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [31]:
#PCR
r2_in_pcr, r2_oos_pcr = [], []
success_ratio_in_pcr, success_ratio_oos_pcr = [], []
feature_importance_pcr = []
best_components_pcr, mse_val_grids_pcr = [], []
df_in_pcr, df_oos_pcr = [], []

max_k = len(covariates)
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    # Tuning
    candidate_ks = range(1, max_k)
    mse_val_grid = []
    best_mse = float('inf')
    best_k = None

    for k in candidate_ks:
        pcr_pipe = Pipeline([
            ('pca', PCA(n_components=k)),
            ('reg', LinearRegression())
        ])
        pcr_pipe.fit(x_train[covariates], y_train)
        y_val_pred = pcr_pipe.predict(x_val[covariates]).ravel()
        mse_val = mean_squared_error(y_val, y_val_pred)
        mse_val_grid.append(mse_val)
        if mse_val < best_mse:
            best_mse = mse_val
            best_k = k

    mse_val_grids_pcr.append(mse_val_grid)
    best_components_pcr.append(best_k)

    # Final fit
    pcr_final = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_final.fit(x_trainval, y_trainval)

    # Importance des variables
    reg_coef = pcr_final.named_steps['reg'].coef_  # (k,)
    pca_components = pcr_final.named_steps['pca'].components_  # (k, n_features)
    projected_coefs = np.abs(reg_coef @ pca_components)  # (n_features,)
    feature_importance_pcr.append(projected_coefs.flatten())

    # Prédictions
    y_trainval_pred = pcr_final.predict(x_trainval).ravel()
    y_test_pred = pcr_final.predict(x_test[covariates]).ravel()

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_pcr.append(r2_in)
    r2_oos_pcr.append(r2_out)
    success_ratio_in_pcr.append(sr_in)
    success_ratio_oos_pcr.append(sr_out)

    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_pcr": y_trainval_pred,
        "Split": split_idx
    })
    df_in_pcr.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_pcr": y_test_pred
    })
    df_oos_pcr.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_pcr = pd.concat(df_in_pcr, ignore_index=True)
df_in_pcr = df_in_pcr.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_pcr = pd.concat(df_oos_pcr, ignore_index=True)

# Dates uniformisées 
df_in_pcr["Date"]  = pd.to_datetime(df_in_pcr["Date"]).dt.to_period("M")
df_oos_pcr["Date"] = pd.to_datetime(df_oos_pcr["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_pcr.shape[0],  "| Duplicates (Date,Ticker) =", df_in_pcr.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_pcr.shape[0], "| Duplicates (Date,Ticker) =", df_oos_pcr.duplicated(["Date","Ticker"]).sum())

  8%|▊         | 1/13 [00:00<00:03,  3.16it/s]

Split 1  R² IN: 0.021073 | OOS: -0.089301 | SR IN: 0.561 | SR OOS: 0.404


 15%|█▌        | 2/13 [00:00<00:03,  3.09it/s]

Split 2  R² IN: 0.016546 | OOS: 0.052764 | SR IN: 0.553 | SR OOS: 0.638


 23%|██▎       | 3/13 [00:01<00:04,  2.49it/s]

...

Split 11  R² IN: 0.020633 | OOS: -0.047067 | SR IN: 0.566 | SR OOS: 0.486


 92%|█████████▏| 12/13 [00:05<00:00,  2.18it/s]

Split 12  R² IN: 0.022146 | OOS: 0.090848 | SR IN: 0.562 | SR OOS: 0.667


100%|██████████| 13/13 [00:05<00:00,  2.36it/s]

Split 13  R² IN: 0.023748 | OOS: 0.031351 | SR IN: 0.565 | SR OOS: 0.573
In sample lines: 20532 | Duplicates (Date,Ticker) = 0
Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [32]:
#ENET
# ElasticNet : Elastic Net
# Hyperparamètres :
# - lambda (alpha) : coefficient de pénalisation choisi pour minimiser la MSE
# - l1_ratio fixé à 0.5

r2_in_en, r2_oos_en = [], []
success_ratio_in_en, success_ratio_oos_en = [], []
y_trainval_en = []

# Hyperparamètres spécifiques
best_lambdas = []
nonzero_counts_en = []
feature_importance_en = []

df_in_en, df_oos_en = [], []

enet_param_grid = {
    'alpha': np.logspace(-4, 0, num=10)
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_lambda = None

    # Recherche du meilleur alpha (lambda)
    for params in ParameterGrid(enet_param_grid):
        enet = ElasticNet(**params, l1_ratio=0.5, max_iter=10000)
        enet.fit(x_train[covariates], y_train)
        y_val_pred = enet.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        if mse < best_mse:
            best_mse = mse
            best_lambda = params['alpha']

    best_lambdas.append(best_lambda)
    print(f"Split {split_idx} : meilleur lambda = {best_lambda}")

    # Entraînement final sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    en_final = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_final.fit(x_trainval, y_trainval)

    y_trainval_pred = en_final.predict(x_trainval)
    y_test_pred = en_final.predict(x_test[covariates])

    # Importance et sparsité
    coefs = np.abs(en_final.coef_)
    feature_importance_en.append(coefs)
    nonzero_count = np.sum(en_final.coef_ != 0)
    nonzero_counts_en.append(nonzero_count)

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_en.append(r2_in)
    r2_oos_en.append(r2_out)
    success_ratio_in_en.append(sr_in)
    success_ratio_oos_en.append(sr_out)
    y_trainval_en.append(y_trainval_pred)

    # Dataframes
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_en": y_trainval_pred,
        "Split": split_idx
    })
    df_in_en.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_en": y_test_pred
    })
    df_oos_en.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_en = pd.concat(df_in_en, ignore_index=True)
df_in_en = df_in_en.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_en = pd.concat(df_oos_en, ignore_index=True)

# Dates uniformisées 
df_in_en["Date"]  = pd.to_datetime(df_in_en["Date"]).dt.to_period("M")
df_oos_en["Date"] = pd.to_datetime(df_oos_en["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_en.shape[0],  "| Duplicates (Date,Ticker) =", df_in_en.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_en.shape[0], "| Duplicates (Date,Ticker) =", df_oos_en.duplicated(["Date","Ticker"]).sum())


  0%|          | 0/13 [00:00<?, ?it/s]

  8%|▊         | 1/13 [00:01<00:16,  1.34s/it]

Split 1 : meilleur lambda = 0.005994842503189409
Split 1  R² IN: 0.019496 | OOS: -0.080538 | SR IN: 0.561 | SR OOS: 0.404


 15%|█▌        | 2/13 [00:04<00:25,  2.34s/it]

Split 2 : meilleur lambda = 0.016681005372000592
Split 2  R² IN: 0.011297 | OOS: 0.037931 | SR IN: 0.552 | SR OOS: 0.636
Split 3 : meilleur lambda = 0.0001


...

 92%|█████████▏| 12/13 [00:38<00:02,  2.51s/it]

Split 12  R² IN: 0.023141 | OOS: 0.089136 | SR IN: 0.561 | SR OOS: 0.668
Split 13 : meilleur lambda = 0.000774263682681127


100%|██████████| 13/13 [00:40<00:00,  3.09s/it]

Split 13  R² IN: 0.024698 | OOS: 0.031522 | SR IN: 0.566 | SR OOS: 0.582
In sample lines: 20532 | Duplicates (Date,Ticker) = 0


Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [33]:
#RF
param_grid_rf = {
    'n_estimators': [200, 400],       
    'max_depth': [3, 4, 5],
    'min_samples_leaf': [3, 5, 10],
    'max_features': ['log2', 1, 2]
}

r2_in_rf, r2_oos_rf = [], []
success_ratio_in_rf, success_ratio_oos_rf = [], []
feature_importance_rf = []
df_in_rf, df_oos_rf = [], []
best_params_rf, mse_val_grids_rf = [], []
y_trainval_rf = []

# Grid Search + Entraînement
for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):
    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_rf):
        rf = RandomForestRegressor(**params, n_jobs=-1, random_state=0)
        rf.fit(x_train[covariates], y_train)
        y_val_pred = rf.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_rf.append(mse_grid)
    best_params_rf.append(best_params)
    print(f"\nSplit {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Entraînement final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    rf_final = RandomForestRegressor(**best_params, n_jobs=-1, random_state=0)
    rf_final.fit(x_trainval, y_trainval)

    y_trainval_pred = rf_final.predict(x_trainval)
    y_test_pred = rf_final.predict(x_test[covariates])

    # R² & Success Ratio
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_rf.append(r2_in)
    r2_oos_rf.append(r2_out)
    success_ratio_in_rf.append(sr_in)
    success_ratio_oos_rf.append(sr_out)
    feature_importance_rf.append(rf_final.feature_importances_)
    y_trainval_rf.append(y_trainval_pred)

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_rf": y_trainval_pred,
        "Split": split_idx
    })
    df_in_rf.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_rf": y_test_pred
    })
    df_oos_rf.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_rf = pd.concat(df_in_rf, ignore_index=True)
df_in_rf = df_in_rf.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_rf = pd.concat(df_oos_rf, ignore_index=True)

# Dates uniformisées 
df_in_rf["Date"]  = pd.to_datetime(df_in_rf["Date"]).dt.to_period("M")
df_oos_rf["Date"] = pd.to_datetime(df_oos_rf["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_rf.shape[0],  "| Duplicates (Date,Ticker) =", df_in_rf.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_rf.shape[0], "| Duplicates (Date,Ticker) =", df_oos_rf.duplicated(["Date","Ticker"]).sum())

  0%|          | 0/13 [00:00<?, ?it/s]


Split 1 : meilleurs params = {'max_depth': 4, 'max_features': 1, 'min_samples_leaf': 5, 'n_estimators': 200} (MSE val = 0.003410)


  8%|▊         | 1/13 [00:41<08:21, 41.78s/it]

Split 1  R² IN: 0.037384 | OOS: -0.085461 | SR IN: 0.561 | SR OOS: 0.404

Split 2 : meilleurs params = {'max_depth': 5, 'max_features': 1, 'min_samples_leaf': 10, 'n_estimators': 200} (MSE val = 0.014989)


 15%|█▌        | 2/13 [01:25<07:52, 42.92s/it]

...

Split 11  R² IN: 0.051459 | OOS: -0.023018 | SR IN: 0.570 | SR OOS: 0.496

Split 12 : meilleurs params = {'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 5, 'n_estimators': 200} (MSE val = 0.006090)


 92%|█████████▏| 12/13 [08:53<00:44, 44.82s/it]

Split 12  R² IN: 0.053750 | OOS: 0.084851 | SR IN: 0.569 | SR OOS: 0.669

Split 13 : meilleurs params = {'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 10, 'n_estimators': 200} (MSE val = 0.005118)


100%|██████████| 13/13 [09:38<00:00, 44.49s/it]

Split 13  R² IN: 0.029325 | OOS: 0.031856 | SR IN: 0.567 | SR OOS: 0.581
In sample lines: 20532 | Duplicates (Date,Ticker) = 0
Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [34]:
#GBRT
from sklearn.ensemble import GradientBoostingRegressor

r2_in_gbrt, r2_oos_gbrt = [], []
success_ratio_in_gbrt, success_ratio_oos_gbrt = [], []
feature_importance_gbrt, complexity_gbrt = [], []
df_in_gbrt, df_oos_gbrt = [], []
y_trainval_gbrt = []

best_params_gbrt = []
mse_val_grids_gbrt = []

param_grid_gbrt = {
    'n_estimators': [300],
    'learning_rate': [0.01],
    'max_depth': [2, 3, 4],
    'loss': ['huber'],
    'alpha': [0.9]
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    best_mse = float('inf')
    best_params = None
    mse_grid = []

    for params in ParameterGrid(param_grid_gbrt):
        model = GradientBoostingRegressor(**params, random_state=0)
        model.fit(x_train[covariates], y_train)
        y_val_pred = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, y_val_pred)
        mse_grid.append((params, mse))
        if mse < best_mse:
            best_mse = mse
            best_params = params

    mse_val_grids_gbrt.append(mse_grid)
    best_params_gbrt.append(best_params)
    print(f"Split {split_idx} — meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Train final
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    gbrt_final = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_final.fit(x_trainval, y_trainval)

    y_trainval_pred = gbrt_final.predict(x_trainval)
    y_test_pred = gbrt_final.predict(x_test[covariates])

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_gbrt.append(r2_in)
    r2_oos_gbrt.append(r2_out)
    success_ratio_in_gbrt.append(sr_in)
    success_ratio_oos_gbrt.append(sr_out)

    # Feature importance
    importances = gbrt_final.feature_importances_
    feature_importance_gbrt.append(importances)
    complexity_gbrt.append(np.sum(importances > 0))

    # DataFrames
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_gbrt": y_trainval_pred,
        "Split": split_idx
    })
    df_in_gbrt.append(df_in_split)

    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_gbrt": y_test_pred
    })
    df_oos_gbrt.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat
df_in_gbrt = pd.concat(df_in_gbrt, ignore_index=True)
df_in_gbrt = df_in_gbrt.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_gbrt = pd.concat(df_oos_gbrt, ignore_index=True)

# Dates uniformisées 
df_in_gbrt["Date"]  = pd.to_datetime(df_in_gbrt["Date"]).dt.to_period("M")
df_oos_gbrt["Date"] = pd.to_datetime(df_oos_gbrt["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_gbrt.shape[0],  "| Duplicates (Date,Ticker) =", df_in_gbrt.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_gbrt.shape[0], "| Duplicates (Date,Ticker) =", df_oos_gbrt.duplicated(["Date","Ticker"]).sum())

  0%|          | 0/13 [00:00<?, ?it/s]

Split 1 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 300} (MSE val = 0.003362)


  8%|▊         | 1/13 [00:53<10:41, 53.46s/it]

Split 1  R² IN: 0.051662 | OOS: -0.068793 | SR IN: 0.575 | SR OOS: 0.419
Split 2 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 3, 'n_estimators': 300} (MSE val = 0.014771)


 15%|█▌        | 2/13 [01:50<10:09, 55.43s/it]

...

 92%|█████████▏| 12/13 [14:28<01:24, 84.08s/it]

Split 12  R² IN: 0.039558 | OOS: 0.081475 | SR IN: 0.572 | SR OOS: 0.669
Split 13 — meilleurs params = {'alpha': 0.9, 'learning_rate': 0.01, 'loss': 'huber', 'max_depth': 2, 'n_estimators': 300} (MSE val = 0.005139)


100%|██████████| 13/13 [16:12<00:00, 74.79s/it]

Split 13  R² IN: 0.029384 | OOS: 0.030122 | SR IN: 0.570 | SR OOS: 0.588


In sample lines: 20532 | Duplicates (Date,Ticker) = 0
Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


In [35]:
#XGBOOST
# Initialisations
r2_in_xgb, r2_oos_xgb = [], []
success_ratio_in_xgb, success_ratio_oos_xgb = [], []
feature_importance_xgb = []
df_in_xgb, df_oos_xgb = [], []
y_trainval_xgb = []

# Grille de recherche
param_grid = {
    'n_estimators': [200, 400],       
    'max_depth': [2, 3],
    'learning_rate': [0.01]
}

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(tqdm(preprocessed_splits), start=1):

    # Recherche du meilleur modèle (validation)
    best_mse = float('inf')
    best_params = None

    for params in ParameterGrid(param_grid):
        model = XGBRegressor(**params, random_state=0, n_jobs=-1)
        model.fit(x_train[covariates], y_train)
        preds = model.predict(x_val[covariates])
        mse = mean_squared_error(y_val, preds)

        if mse < best_mse:
            best_mse = mse
            best_params = params

    print(f"Split {split_idx} : meilleurs params = {best_params} (MSE val = {best_mse:.6f})")

    # Réentraînement sur train + val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])
    tickers_trainval = pd.concat([x_train["Ticker"], x_val["Ticker"]], ignore_index=True)
    dates_trainval = pd.concat([x_train["Date"], x_val["Date"]], ignore_index=True)

    model = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    model.fit(x_trainval, y_trainval)

    y_trainval_pred = model.predict(x_trainval)
    y_test_pred = model.predict(x_test[covariates])

    # Scores
    r2_in = r2(y_trainval, y_trainval_pred)
    r2_out = r2(y_test, y_test_pred)
    sr_in = success_ratio(y_trainval, y_trainval_pred)
    sr_out = success_ratio(y_test, y_test_pred)

    r2_in_xgb.append(r2_in)
    r2_oos_xgb.append(r2_out)
    success_ratio_in_xgb.append(sr_in)
    success_ratio_oos_xgb.append(sr_out)

    # Importance des variables
    importances = model.feature_importances_
    feature_importance_xgb.append(importances)

    # Prédictions in-sample
    df_in_split = pd.DataFrame({
        "Date": dates_trainval,
        "Ticker": tickers_trainval,
        "y_true": y_trainval.values,
        "y_pred_xgb": y_trainval_pred,
        "Split": split_idx
    })
    df_in_xgb.append(df_in_split)

    # Prédictions OOS
    df_test_split = pd.DataFrame({
        "Split": split_idx,
        "Date": x_test["Date"].reset_index(drop=True),
        "Ticker": x_test["Ticker"].reset_index(drop=True),
        "y_true": y_test.reset_index(drop=True),
        "y_pred_xgb": y_test_pred
    })
    df_oos_xgb.append(df_test_split)

    print(f"Split {split_idx}  R² IN: {r2_in:.6f} | OOS: {r2_out:.6f} | SR IN: {sr_in:.3f} | SR OOS: {sr_out:.3f}")

# Concat final
df_in_xgb = pd.concat(df_in_xgb, ignore_index=True)
df_in_xgb = df_in_xgb.sort_values("Split").drop_duplicates(["Date","Ticker"], keep="first")
df_oos_xgb = pd.concat(df_oos_xgb, ignore_index=True)

# Dates uniformisées 
df_in_xgb["Date"]  = pd.to_datetime(df_in_xgb["Date"]).dt.to_period("M")
df_oos_xgb["Date"] = pd.to_datetime(df_oos_xgb["Date"]).dt.to_period("M")

#Checks 
print("In sample lines:",  df_in_xgb.shape[0],  "| Duplicates (Date,Ticker) =", df_in_xgb.duplicated(["Date","Ticker"]).sum())
print("Out-of-sample lines:",  df_oos_xgb.shape[0], "| Duplicates (Date,Ticker) =", df_oos_xgb.duplicated(["Date","Ticker"]).sum())

  0%|          | 0/13 [00:00<?, ?it/s]

Split 1 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 200} (MSE val = 0.003460)


  8%|▊         | 1/13 [00:02<00:34,  2.89s/it]

Split 1  R² IN: 0.053339 | OOS: -0.103198 | SR IN: 0.566 | SR OOS: 0.411
Split 2 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 2, 'n_estimators': 200} (MSE val = 0.015148)


 15%|█▌        | 2/13 [00:04<00:25,  2.34s/it]

...

Split 11  R² IN: 0.028520 | OOS: -0.024987 | SR IN: 0.567 | SR OOS: 0.486
Split 12 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 400} (MSE val = 0.006056)


 92%|█████████▏| 12/13 [00:29<00:02,  2.69s/it]

Split 12  R² IN: 0.056809 | OOS: 0.097231 | SR IN: 0.573 | SR OOS: 0.668
Split 13 : meilleurs params = {'learning_rate': 0.01, 'max_depth': 3, 'n_estimators': 400} (MSE val = 0.005064)


100%|██████████| 13/13 [00:32<00:00,  2.50s/it]

Split 13  R² IN: 0.058546 | OOS: 0.047074 | SR IN: 0.575 | SR OOS: 0.592
In sample lines: 20532 | Duplicates (Date,Ticker) = 0
Out-of-sample lines: 9204 | Duplicates (Date,Ticker) = 0


III. Results metrics (R2, Success ratios)

In [36]:
#Découpage de HA

df_in_ha = df_ha[(df_ha["Date"] >= "1990-12") & (df_ha["Date"] <= "2019-11")].copy()
df_oos_ha = df_ha[(df_ha["Date"] >= "2007-12") & (df_ha["Date"] <= "2020-11")].copy()
df_in_ha["Date"]  = pd.to_datetime(df_in_ha["Date"]).dt.to_period("M")
df_oos_ha["Date"] = pd.to_datetime(df_oos_ha["Date"]).dt.to_period("M")

#Vérif
print("HA In-sample :", df_in_ha["Date"].min(), "→", df_in_ha["Date"].max())
print("HA OOS       :", df_oos_ha["Date"].min(), "→", df_oos_ha["Date"].max())
print("In-sample OLS :", df_in_ols["Date"].min(), "→", df_in_ols["Date"].max())
print("OOS OLS      :", df_oos_ols["Date"].min(), "→", df_oos_ols["Date"].max())
print("df_in_ha   :", len(df_in_ha))
print("df_oos_ha  :", len(df_oos_ha))
print("df_in_ols  :", len(df_in_ols))
print("df_oos_ols :", len(df_oos_ols))

HA In-sample : 1990-12 → 2019-11
HA OOS       : 2007-12 → 2020-11
In-sample OLS : 1990-12 → 2019-11
OOS OLS      : 2007-12 → 2020-11
df_in_ha   : 20532
df_oos_ha  : 9204
df_in_ols  : 20532
df_oos_ols : 9204


In [37]:
#R² - Gu et al. 

models = {
    "OLS":  ("df_in_ols",  "y_pred_ols"),
    "PLS":  ("df_in_pls",  "y_pred_pls"),
    "PCR":  ("df_in_pcr",  "y_pred_pcr"),
    "Enet": ("df_in_en", "y_pred_en"),
    "RF":   ("df_in_rf",   "y_pred_rf"),
    "GBRT": ("df_in_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_in_xgb",  "y_pred_xgb")
}

r2_results_in = {}
for model, (df_name, pred_col) in models.items():
    df = globals()[df_name]   # récupère la variable df_in_xxx par son nom
    r2_val = r2(df["y_true"], df[pred_col])
    r2_results_in[model] = r2_val

print(r2_results_in)

models_oos = {
    "OLS":  ("df_oos_ols",  "y_pred_ols"),
    "PLS":  ("df_oos_pls",  "y_pred_pls"),
    "PCR":  ("df_oos_pcr",  "y_pred_pcr"),
    "Enet": ("df_oos_en", "y_pred_en"),
    "RF":   ("df_oos_rf",   "y_pred_rf"),
    "GBRT": ("df_oos_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_oos_xgb",  "y_pred_xgb")
}

r2_results_oos = {}

for model, (df_name, pred_col) in models_oos.items():
    df = globals()[df_name]   # car tu stockes des morceaux dans une liste
    r2_val = r2(df["y_true"], df[pred_col])
    r2_results_oos[model] = float(r2_val)

print(r2_results_oos)

# IN-SAMPLE — R² brut de HA
tmp_in = (
    df_in_ols[["Date","Ticker","y_true"]]
    .merge(df_in_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
r2_ha_in = float(r2(tmp_in["y_true"], tmp_in["ha_return"]))
print("R²_in (HA) :", r2_ha_in)

# OUT-OF-SAMPLE — R² brut de HA
tmp_oos = (
    df_oos_ols[["Date","Ticker","y_true"]]
    .merge(df_oos_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
r2_ha_oos = float(r2(tmp_oos["y_true"], tmp_oos["ha_return"]))
print("R²_oos (HA):", r2_ha_oos)

# Option : ajoute-les à tes dicts de résultats pour les tables
r2_results_in["HA"]  = r2_ha_in
r2_results_oos["HA"] = r2_ha_oos

{'OLS': np.float64(0.027609610289898723), 'PLS': np.float64(0.02560745550956589), 'PCR': np.float64(0.024112224393950443), 'Enet': np.float64(0.022995329340521287), 'RF': np.float64(0.04578830724342475), 'GBRT': np.float64(0.054893901826133074), 'XGB': np.float64(0.05831082598956494)}
{'OLS': 0.015927667360057463, 'PLS': 0.017173005510793038, 'PCR': 0.017102597907013073, 'Enet': 0.017272670765217657, 'RF': 0.01911265085716729, 'GBRT': 0.021306072544528543, 'XGB': 0.017110390306355416}
R²_in (HA) : 0.011172159310847252
R²_oos (HA): 0.0149254641621418


In [38]:
#R² - Benchmark HA - Xiu et Liu 

#R2 in
df_in_ha_bench = df_in_ha[["Date", "Ticker", "ha_return"]]

r2_vs_ha_in = {}

for model, (df_name, pred_col) in models.items():
    df = globals()[df_name]
    tmp = df.merge(df_in_ha_bench, on=["Date", "Ticker"], how="inner") #permet de comparer les lignes ayant les mêmes dates et tickers
    r2_vs_ha_in[model] = float(
        r2_vs_benchmark(tmp["y_true"].to_numpy(),
                        tmp[pred_col].to_numpy(),
                        tmp["ha_return"].to_numpy())
    )

print(r2_vs_ha_in)

r2_vs_ha_oos = {}
df_oos_ha_bench = df_oos_ha[["Date", "Ticker", "ha_return"]]

for model, (df_name, pred_col) in models_oos.items():
    df = globals()[df_name]
    # Alignement sur Date + Ticker
    tmp = df.merge(df_oos_ha_bench, on=["Date", "Ticker"], how="inner")
    # R² vs benchmark HA
    r2_vs_ha_oos[model] = float(
        r2_vs_benchmark(tmp["y_true"].to_numpy(),
                        tmp[pred_col].to_numpy(),
                        tmp["ha_return"].to_numpy())
    )

print(r2_vs_ha_oos)

{'OLS': 0.016623167656359072, 'PLS': 0.014598391757111084, 'PCR': 0.013086266942165192, 'Enet': 0.011956752776533852, 'RF': 0.03500725455752962, 'GBRT': 0.04421572766884707, 'XGB': 0.04767125756275714}
{'OLS': 0.0010173881888675407, 'PLS': 0.00228159521628446, 'PCR': 0.0022101208240241643, 'Enet': 0.0023827705596708437, 'RF': 0.004250629310465381, 'GBRT': 0.006477284865515021, 'XGB': 0.002218031290754263}


In [39]:
#Success ratio - Xiu and Liu 
success_in = {}
for model, (df_name, pred_col) in {
    "OLS":  ("df_in_ols",  "y_pred_ols"),
    "PLS":  ("df_in_pls",  "y_pred_pls"),
    "PCR":  ("df_in_pcr",  "y_pred_pcr"),
    "Enet": ("df_in_en",   "y_pred_en"),
    "RF":   ("df_in_rf",   "y_pred_rf"),
    "GBRT": ("df_in_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_in_xgb",  "y_pred_xgb")
}.items():
    df = globals()[df_name]
    success_in[model] = float(success_ratio(df["y_true"], df[pred_col]))

print("Success ratio IN:", success_in)

success_oos = {}
for model, (df_name, pred_col) in {
    "OLS":  ("df_oos_ols",  "y_pred_ols"),
    "PLS":  ("df_oos_pls",  "y_pred_pls"),
    "PCR":  ("df_oos_pcr",  "y_pred_pcr"),
    "Enet": ("df_oos_en",   "y_pred_en"),
    "RF":   ("df_oos_rf",   "y_pred_rf"),
    "GBRT": ("df_oos_gbrt", "y_pred_gbrt"),
    "XGB":  ("df_oos_xgb",  "y_pred_xgb")
}.items():
    df = globals()[df_name]
    success_oos[model] = float(success_ratio(df["y_true"], df[pred_col]))

print("Success ratio OOS:", success_oos)

# R2 in sample HA
tmp_in = (
    df_in_ols[["Date","Ticker","y_true"]]
    .merge(df_in_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
sr_ha_in = float(success_ratio(tmp_in["y_true"], tmp_in["ha_return"]))

# R2 oos HA
tmp_oos = (
    df_oos_ols[["Date","Ticker","y_true"]]
    .merge(df_oos_ha[["Date","Ticker","ha_return"]], on=["Date","Ticker"], how="inner")
)
sr_ha_oos = float(success_ratio(tmp_oos["y_true"], tmp_oos["ha_return"]))

print({"HA_in": sr_ha_in, "HA_oos": sr_ha_oos})

Success ratio IN: {'OLS': 0.5640463666471849, 'PLS': 0.5671147477108903, 'PCR': 0.5669686343269044, 'Enet': 0.5662380674069745, 'RF': 0.5688194038573934, 'GBRT': 0.5812390414962011, 'XGB': 0.572423533995714}
Success ratio OOS: {'OLS': 0.5643198609300304, 'PLS': 0.5696436332029552, 'PCR': 0.5714906562364189, 'Enet': 0.5718166014776185, 'RF': 0.5769230769230769, 'GBRT': 0.5767057800956106, 'XGB': 0.5750760538896132}
{'HA_in': 0.5646308201831288, 'HA_oos': 0.575619295958279}


In [40]:
#Création des tables latex 
models_list = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB", "HA"]

#Table 1 : R²-in sample brut, r2-IS vs HA, Success ratio in sample 
rows = []
for model in models_list:
    if model == "HA":
        r2_in = r2_ha_in          
        r2_vs_ha = 0.0           
        sr_in = sr_ha_in
    else:
        r2_in = r2_results_in[model]
        r2_vs_ha = r2_vs_ha_in[model]
        sr_in = success_in[model]

    rows.append({
        "Model": model,
        "In-sample $R^2$": r2_in,
        "In-sample $R^2$ vs HA": r2_vs_ha,
        "In-sample Success Ratio": sr_in
    })

df_table1 = pd.DataFrame(rows)

# Mise en forme
for col in ["In-sample $R^2$", "In-sample $R^2$ vs HA", "In-sample Success Ratio"]:
    df_table1[col] = (df_table1[col].astype(float) * 100).apply(
        lambda x: f"{x:.2f}" if not np.isnan(x) else ""
    )

latex_table1 = df_table1.to_latex(index=False, escape=False, column_format="lccc")
print(latex_table1)

#Table 2 : R2 in sample / R2 out of sample (Gu et al. comparaison)
models_list_noha = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB"]

rows = []
for model in models_list_noha:
    r2_in  = r2_results_in[model]
    r2_oos = r2_results_oos[model]
    rows.append({
        "Model": model,
        "In-sample $R^2$": r2_in,
        "Out-of-sample $R^2$": r2_oos
    })

df_table2 = pd.DataFrame(rows)

# Mise en forme : *100 et arrondi à 2 décimales
for col in ["In-sample $R^2$", "Out-of-sample $R^2$"]:
    df_table2[col] = (df_table2[col].astype(float) * 100).apply(lambda x: f"{x:.2f}")

latex_table2 = df_table2.to_latex(index=False, escape=False, column_format="lcc")
print(latex_table2)

# Table 3 : R² vs HA (OOS) + Success Ratio OOS, avec HA
models_all = ["OLS", "PLS", "PCR", "Enet", "RF", "GBRT", "XGB", "HA"]

rows = []
for model in models_all:
    if model == "HA":
        r2_vs_ha = 0.0              # par définition, HA vs HA = 0
        sr_oos   = sr_ha_oos        # success ratio de HA
    else:
        r2_vs_ha = float(r2_vs_ha_oos[model])
        sr_oos   = float(success_oos[model])

    rows.append({
        "Model": model,
        "Out-of-sample $R^2$ vs HA": r2_vs_ha,
        "Out-of-sample Success Ratio": sr_oos
    })

df_table3 = pd.DataFrame(rows)

# Mise en forme : *100 et arrondi à 2 décimales
for col in ["Out-of-sample $R^2$ vs HA", "Out-of-sample Success Ratio"]:
    df_table3[col] = (df_table3[col].astype(float) * 100).apply(lambda x: f"{x:.2f}")

latex_table3 = df_table3.to_latex(index=False, escape=False, column_format="lcc")
print(latex_table3)


\begin{tabular}{lccc}
\toprule
Model & In-sample $R^2$ & In-sample $R^2$ vs HA & In-sample Success Ratio \\
\midrule
OLS & 2.76 & 1.66 & 56.40 \\
PLS & 2.56 & 1.46 & 56.71 \\
PCR & 2.41 & 1.31 & 56.70 \\
Enet & 2.30 & 1.20 & 56.62 \\
RF & 4.58 & 3.50 & 56.88 \\
GBRT & 5.49 & 4.42 & 58.12 \\
XGB & 5.83 & 4.77 & 57.24 \\
HA & 1.12 & 0.00 & 56.46 \\
\bottomrule
\end{tabular}

\begin{tabular}{lcc}
\toprule
Model & In-sample $R^2$ & Out-of-sample $R^2$ \\
\midrule
OLS & 2.76 & 1.59 \\
PLS & 2.56 & 1.72 \\
PCR & 2.41 & 1.71 \\
Enet & 2.30 & 1.73 \\
RF & 4.58 & 1.91 \\
GBRT & 5.49 & 2.13 \\
XGB & 5.83 & 1.71 \\
\bottomrule
\end{tabular}

\begin{tabular}{lcc}
\toprule
Model & Out-of-sample $R^2$ vs HA & Out-of-sample Success Ratio \\
\midrule
OLS & 0.10 & 56.43 \\
PLS & 0.23 & 56.96 \\
PCR & 0.22 & 57.15 \\
Enet & 0.24 & 57.18 \\
RF & 0.43 & 57.69 \\
GBRT & 0.65 & 57.67 \\
XGB & 0.22 & 57.51 \\
HA & 0.00 & 57.56 \\
\bottomrule
\end{tabular}



Diebold test 

In [41]:
#Vérif : y true bons
models = {
    "OLS": df_oos_ols,
    "PLS": df_oos_pls,
    "PCR": df_oos_pcr,
    "Enet": df_oos_en,
    "RF": df_oos_rf,
    "GBRT": df_oos_gbrt,
    "XGB": df_oos_xgb,
}

# choisir une référence, par ex OLS
ref_name, ref_df = "OLS", models["OLS"]

for name, df in models.items():
    if name == ref_name:
        continue
    check = ref_df.merge(df, on=["Date","Ticker"], suffixes=("_ref", f"_{name}"))
    mismatches = (check["y_true_ref"] != check[f"y_true_{name}"]).sum()
    print(f"Différences de y_true entre {ref_name} et {name} :", mismatches)

Différences de y_true entre OLS et PLS : 0
Différences de y_true entre OLS et PCR : 0
Différences de y_true entre OLS et Enet : 0
Différences de y_true entre OLS et RF : 0
Différences de y_true entre OLS et GBRT : 0
Différences de y_true entre OLS et XGB : 0


In [42]:
#Créations tableaux prédictions
df_oos = df_oos_ols.copy()[["Date","Ticker","y_true","y_pred_ols"]]

models = {
    "PLS": (df_oos_pls, "y_pred_pls"),
    "PCR": (df_oos_pcr, "y_pred_pcr"),
    "EN": (df_oos_en, "y_pred_en"),
    "RF": (df_oos_rf, "y_pred_rf"),
    "GBRT": (df_oos_gbrt, "y_pred_gbrt"),
    "XGB": (df_oos_xgb, "y_pred_xgb"),
    "HA": (df_oos_ha, "ha_return")
}

for name, (df_model, col_pred) in models.items():
    df_oos = df_oos.merge(
        df_model[["Date","Ticker",col_pred]].rename(columns={col_pred:name}),
        on=["Date","Ticker"],
        how="left"
    )

df_oos.rename(columns={"y_pred_ols": "OLS"}, inplace=True)

print(df_oos.head())

      Date Ticker    y_true       OLS       PLS       PCR        EN        RF  \
0  2007-12    MRK -0.208777 -0.008009  0.007752  0.011183  0.012056  0.011290   
1  2007-12    VMC -0.012720  0.028163  0.021860  0.016804  0.012234  0.022986   
2  2007-12    ETN -0.145061  0.015774  0.017239  0.017025  0.012948  0.012838   
3  2007-12    DHR -0.147757  0.022531  0.012987  0.011361  0.014375  0.013145   
4  2007-12    ITW -0.060373  0.022524  0.013219  0.011648  0.013007  0.012236   

       GBRT       XGB        HA  
0  0.002507  0.006095  0.008897  
1 -0.004891  0.023920  0.009470  
2  0.005638  0.007881  0.009723  
3  0.013645  0.015040  0.018305  
4  0.015063  0.013648  0.010440  


In [43]:
#DIABOLD TEST 

df_diebold = df_oos[["y_true","OLS","PLS","PCR","EN","RF","GBRT","XGB","HA"]].copy()

results = []

models = [c for c in df_diebold.columns if c not in ["y_true","Date","Ticker"]]

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_diebold['y_true'] - df_diebold[model1])**2
        e2 = (df_diebold['y_true'] - df_diebold[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })


dm_df = pd.DataFrame(results)
dm_df_sorted = dm_df.sort_values(by="P-Value", ascending=True)
#print(dm_df_sorted)
print(dm_df)

   Model 1 Model 2   DM Stat   P-Value Best Model
0      OLS     PLS  0.762172  0.445977        PLS
1      OLS     PCR  0.591966  0.553888        PCR
2      OLS      EN  0.544256  0.586278         EN
3      OLS      RF  1.369272  0.170948         RF
4      OLS    GBRT  2.117521  0.034242       GBRT
5      OLS     XGB  0.372704  0.709378        XGB
6      OLS      HA -0.307460  0.758500        OLS
7      PLS     PCR -0.086648  0.930953        PLS
8      PLS      EN  0.059088  0.952883         EN
9      PLS      RF  1.250618  0.211106         RF
10     PLS    GBRT  1.973339  0.048487       GBRT
11     PLS     XGB -0.022055  0.982404        PLS
12     PLS      HA -0.919616  0.357797        PLS
13     PCR      EN  0.122714  0.902336         EN
14     PCR      RF  1.378846  0.167976         RF
15     PCR    GBRT  2.038320  0.041546       GBRT
16     PCR     XGB  0.002671  0.997869        XGB
17     PCR      HA -0.998997  0.317822        PCR
18      EN      RF  1.210125  0.226262         RF


Portfolios

In [48]:
#I. PORTEFEUILLES EQUALLY WEIGHT
pred_cols = ['OLS','PLS','PCR','EN', 'RF','GBRT','XGB','HA']
col_ret = 'y_true'
nb_groups = 3  
final_table = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_oos.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in range(nb_groups):
            sub = i[i["group"] == grp]
            mean_pred = sub[col_pred].mean()
            mean_real = sub[col_ret].mean()
            std_real = sub[col_ret].std()
            sr_real = mean_real / std_real if std_real > 1e-6 else np.nan

            all_rows.append({
                "Date": date,
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    df_result = pd.DataFrame(all_rows)

    df_grouped = df_result.groupby("Group").agg({
        "Pred": "mean",
        "Avg": "mean",
        "SD": "mean"
    }).reset_index()

    # Recalcul SR correctement après aggregation
    df_grouped["SR"] = df_grouped["Avg"] / df_grouped["SD"]
    
    # Calcul SR H-L robuste
    avg_high = df_grouped.loc[nb_groups-1, "Avg"]
    avg_low  = df_grouped.loc[0, "Avg"]
    sd_high  = df_grouped.loc[nb_groups-1, "SD"]
    sd_low   = df_grouped.loc[0, "SD"]

    hl_avg = avg_high - avg_low
    hl_sd = np.sqrt(sd_high**2 + sd_low**2)
    hl_sr = hl_avg / hl_sd if hl_sd > 1e-6 else np.nan

    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  hl_avg,
        "SD":   hl_sd,
        "SR":   hl_sr
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table[col_pred.upper()] = df_grouped

# Affichage
for model, df in final_table.items():
    print(f"\n{model}")
    print(df.to_string(index=False))


OLS
Group     Pred      Avg       SD       SR
    0 0.000970 0.005028 0.052686 0.095441
    1 0.011053 0.011469 0.053178 0.215679
    2 0.022535 0.016074 0.066351 0.242253
  H-L 0.021565 0.011045 0.084724 0.130367

PLS
Group     Pred      Avg       SD       SR
    0 0.003783 0.006384 0.049103 0.130017
    1 0.011112 0.010850 0.050951 0.212953
    2 0.019938 0.015407 0.069193 0.222669
  H-L 0.016155 0.009023 0.084846 0.106344

PCR
Group     Pred      Avg       SD       SR
    0 0.004707 0.007196 0.047728 0.150780
    1 0.011269 0.011570 0.052846 0.218930
    2 0.018959 0.014021 0.068554 0.204530
  H-L 0.014253 0.006825 0.083532 0.081703

EN
Group     Pred      Avg       SD       SR
    0 0.007342 0.006891 0.054828 0.125690
    1 0.011513 0.011484 0.054196 0.211901
    2 0.016354 0.014375 0.064914 0.221442
  H-L 0.009012 0.007483 0.084970 0.088071

RF
Group     Pred      Avg       SD       SR
    0 0.008075 0.004722 0.050171 0.094127
    1 0.010739 0.010224 0.050646 0.201869
    2 0.016

KeyError: "['Mkt_Cap_Monthly'] not in index"

In [ ]:
df_portfolio_me = pd.read_excel("me_df.xlsx")
df_portfolio_me = df_portfolio_me.sort_values("Date").reset_index(drop=True) 
df_portfolio_me["Date"]  = pd.to_datetime(df_portfolio_me["Date"]).dt.to_period("M")
df_portfolio_me = df_portfolio_me.drop(columns=["Unnamed: 0"])

df_portfolio_me = df_portfolio_me[(df_portfolio_me["Date"] >= "1990-12") & (df_portfolio_me["Date"] <= "2019-11")].copy()
df_portfolio_me = df_portfolio_me[(df_portfolio_me["Date"] >= "2007-12") & (df_portfolio_me["Date"] <= "2020-11")].copy()

df_pw = df_oos.merge(df_portfolio_me, on=["Date","Ticker"], how="left")
df_pw = df_pw.rename(columns={"Mkt_Cap_Monthly": "mktcap"})


In [46]:
# 2. Portefeuilles market-weighted
pred_cols = ['OLS','PLS','PCR','EN', 'RF','GBRT','XGB','HA']
col_ret = 'y_true'
nb_groups = 3  
final_table_vw = {}

for col_pred in pred_cols:
    all_rows = []

    for date, group in df_pw.groupby('Date'):
        group = group.sort_values(col_pred).reset_index(drop=True)
        n = len(group)
        size = n // nb_groups
        group["group"] = [min(i // size, nb_groups - 1) for i in range(n)]

        for grp in range(nb_groups):
            sub = group[group["group"] == grp]
            if sub["mktcap"].sum() == 0:
                continue
            weights = sub["mktcap"] / sub["mktcap"].sum()
            mean_pred = (weights * sub[col_pred]).sum()
            mean_real = (weights * sub[col_ret]).sum()
            std_real = np.sqrt((weights * (sub[col_ret] - mean_real) ** 2).sum())
            sr_real = mean_real / std_real if std_real > 1e-6 else np.nan

            all_rows.append({
                "Date": date,
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    df_result = pd.DataFrame(all_rows)

    # Moyenne des valeurs par groupe (0,1,2) sur toutes les dates
    df_grouped = df_result.groupby("Group").agg({
        "Pred": "mean",
        "Avg": "mean",
        "SD": "mean",
        "SR": "mean"
    }).reset_index()

    # Calcul SR H-L robuste
    avg_high = df_grouped.loc[nb_groups-1, "Avg"]
    avg_low  = df_grouped.loc[0, "Avg"]
    sd_high  = df_grouped.loc[nb_groups-1, "SD"]
    sd_low   = df_grouped.loc[0, "SD"]

    hl_avg = avg_high - avg_low
    hl_sd = np.sqrt(sd_high**2 + sd_low**2)
    hl_sr = hl_avg / hl_sd if hl_sd > 1e-6 else np.nan

    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  hl_avg,
        "SD":   hl_sd,
        "SR":   hl_sr
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table_vw[col_pred.upper()] = df_grouped

# Affichage final des résultats market-weighted
for model, df in final_table_vw.items():
    print(f"\n{model} (Market Weighted)")
    print(df.to_string(index=False))



OLS (Market Weighted)
Group      Pred      Avg       SD       SR
    0 -0.000072 0.005210 0.046284 0.176819
    1  0.010691 0.010348 0.045972 0.237760
    2  0.021263 0.014681 0.053650 0.281204
  H-L  0.021335 0.009471 0.070855 0.133663

PLS (Market Weighted)
Group     Pred      Avg       SD       SR
    0 0.002813 0.005371 0.045151 0.160382
    1 0.010824 0.011791 0.045200 0.302581
    2 0.018523 0.011509 0.056979 0.211837
  H-L 0.015710 0.006139 0.072699 0.084437

PCR (Market Weighted)
Group     Pred      Avg       SD       SR
    0 0.003881 0.006608 0.043432 0.174621
    1 0.010985 0.010716 0.048259 0.263461
    2 0.017744 0.009124 0.056432 0.193340
  H-L 0.013863 0.002516 0.071210 0.035332

EN (Market Weighted)
Group     Pred      Avg       SD       SR
    0 0.007109 0.004458 0.046165 0.165364
    1 0.011400 0.010344 0.045315 0.242890
    2 0.015499 0.012252 0.051680 0.283104
  H-L 0.008391 0.007795 0.069297 0.112482

RF (Market Weighted)
Group     Pred      Avg       SD       SR


In [ ]:
# === TABLE 8 : Drawdowns, Turnover, Risk-adjusted Performance ===
import numpy as np
import pandas as pd

pred_cols = ['OLS','PLS','PCR','EN','RF','GBRT','XGB','HA']
ret_col   = 'y_true'
nb_groups = 3

def build_portfolio(df, signal_col, weighting="equal"):
    """Construit séries r_low, r_high, r_HL selon weighting ('equal' ou 'value')."""
    rows = []
    w_by_date = {}
    for date, g in df.groupby('Date'):
        g = g.sort_values(signal_col).reset_index(drop=True)
        n = len(g)
        size = max(1, n // nb_groups)
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        g['group'] = groups

        low  = g[g['group'] == 0]
        high = g[g['group'] == nb_groups - 1]

        if weighting == "equal":
            r_low  = low[ret_col].mean()
            r_high = high[ret_col].mean()
            w = pd.Series(0.0, index=g['Ticker'])
            if len(high)>0: w.loc[high['Ticker']] =  1/len(high)
            if len(low) >0: w.loc[low['Ticker']]  = -1/len(low)

        elif weighting == "value":
            
            low_w  = low['mktcap'] / low['mktcap'].sum() if len(low)>0 else []
            high_w = high['mktcap'] / high['mktcap'].sum() if len(high)>0 else []
            r_low  = (low[ret_col]*low_w).sum()  if len(low)>0 else np.nan
            r_high = (high[ret_col]*high_w).sum() if len(high)>0 else np.nan
            w = pd.Series(0.0, index=g['Ticker'])
            if len(high)>0: w.loc[high['Ticker']] =  high_w.values
            if len(low) >0: w.loc[low['Ticker']]  = -low_w.values

        r_hl = r_high - r_low
        rows.append({'Date': date, 'r_HL': r_hl})
        w_by_date[date] = w

    df_s = pd.DataFrame(rows).sort_values('Date').reset_index(drop=True)
    return df_s, w_by_date

def max_drawdown(r):
    cum = (1 + r).cumprod()
    dd = cum / cum.cummax() - 1
    return dd.min()

def turnover_series(w_by_date):
    dates = sorted(w_by_date.keys())
    out = []
    for t in range(1, len(dates)):
        d0, d1 = dates[t-1], dates[t]
        w0 = w_by_date[d0]
        w1 = w_by_date[d1]
        all_idx = w0.index.union(w1.index)
        w0a = w0.reindex(all_idx).fillna(0.0)
        w1a = w1.reindex(all_idx).fillna(0.0)
        out.append(0.5 * (w1a - w0a).abs().sum())
    return np.mean(out) if out else np.nan

def compute_table(weighting="equal"):
    rows = []
    for model in pred_cols:
        df_s, w_by_date = build_portfolio(df_oos[['Date','Ticker',ret_col,model,'mktcap']],
                                          signal_col=model, weighting=weighting)
        r = df_s['r_HL'].dropna()
        mdd   = max_drawdown(r) * 100   # %
        max1m = r.min() * 100           # %
        tov   = turnover_series(w_by_date) * 100  # %
        rows.append([model.upper(), mdd, max1m, tov])
    return pd.DataFrame(rows, columns=["Model","Max DD(%)","Max 1M Loss(%)","Turnover(%)"])

# Table 8 - Value Weighted
table8_vw = compute_table("value")
print("\nTable 8: Drawdowns and Turnover (Value Weighted)")
print(table8_vw.to_string(index=False))

# Table 8 - Equally Weighted
table8_ew = compute_table("equal")
print("\nTable 8: Drawdowns and Turnover (Equally Weighted)")
print(table8_ew.to_string(index=False))


KeyError: "['mktcap'] not in index"

RESTE / CE QUI SUIT → à trier / code faux 

//////////////////////////////////////////////////////////////////////

III. RESULTS

CALCULS R²

In [47]:
import matplotlib.pyplot as plt
import numpy as np

# Dictionnaire des modèles avec leurs R² OOS
r2_models = {
    "HA": r2_oos_ha,
    "OLS": r2_oos_ols,
    "PLS": r2_oos_pls,
    "ENET": r2_oos_en,
    "PCR": r2_oos_pcr,
    "RF": r2_oos_rf,
    "GBRT": r2_oos_gbrt,
    "XGB": r2_oos_xgb
}

fig, axs = plt.subplots(2, 4, figsize=(16, 6))
axs = axs.flatten()

for i, (model_name, r2_oos) in enumerate(r2_models.items()):
    axs[i].plot(first_date_split, r2_oos, marker='o')
    axs[i].axhline(np.mean(r2_oos), color='red', linestyle='--', linewidth=1)
    axs[i].set_title(f"{model_name} (mean: {round(np.mean(r2_oos), 3)})", fontsize=10)
    axs[i].set_xticks(first_date_split)
    axs[i].tick_params(axis='x', rotation=45)
    axs[i].set_ylim(-0.1, 0.15)
    axs[i].set_yticks(np.arange(-0.1, 0.175, 0.025)) 
    axs[i].grid(True)

plt.tight_layout()
plt.suptitle("R² out-of-sample by split", fontsize=14, y=1.05)
plt.show()


NameError: name 'r2_oos_ha' is not defined

In [ ]:
#R2 BENCHMARK IN-SAMPLE : 
def r2_vs_benchmark(y_true, y_pred_model, y_pred_bench):
    T = len(y_true)
    mspe_model = (1/T) * np.sum((y_true - y_pred_model)**2)
    mspe_bench = (1/T) * np.sum((y_true - y_pred_bench)**2)
    return 1 - (mspe_model / mspe_bench)


In [ ]:
df_oos = pd.DataFrame(predictions_oos)
df_oos["y_true"] = y_true  # ajoute la colonne avec les vraies valeurs


#DIABOLD TEST 
models = [col for col in df_oos.columns if col != "y_true"]


results = []

for i in range(len(models)):
    for j in range(i+1, len(models)):
        model1 = models[i]
        model2 = models[j]

        #calcule mse du modèle 1 et 2 
        e1 = (df_oos['y_true'] - df_oos[model1])**2
        e2 = (df_oos['y_true'] - df_oos[model2])**2
        d = e1 - e2
        d = d.dropna()

        X = np.ones(len(d))  # régression constante
        model = sm.OLS(d, X).fit()
        cov = cov_hac(model, nlags=1)  
        se = np.sqrt(cov[0][0])
        dm_stat = d.mean() / se
        p_value = 2 * (1 - t.cdf(abs(dm_stat), df=len(d) - 1))

        results.append({
            'Model 1': model1,
            'Model 2': model2,
            'DM Stat': dm_stat,
            'P-Value': p_value,
            'Best Model': model2 if dm_stat > 0 else model1
        })


dm_df = pd.DataFrame(results)
dm_df_sorted = dm_df.sort_values(by="P-Value", ascending=True)
print(dm_df_sorted)
#dm_df.sort_values(by="P-Value", ascending=True, inplace=True)
print(dm_df)

In [ ]:
print(df_oos)

COMPLEXITE MODELES

In [ ]:
#GRAPH → plot complexités des modèles 

plt.rcParams["text.usetex"] = False #oblige matplot lib à ne pas utiliser latex
plt.rcParams["font.family"] = "Arial" #met arial pcq il me demandait palatino ?

dates_splits = pd.to_datetime(first_date_split)

#Récupérer les paramètres de chaque split 
max_depths_rf = [d["max_depth"] for d in best_params_rf]

print(dates_splits)

df_best_param = pd.DataFrame({
    "Date": dates_splits,
    "Best_k_pls": best_components_pls,
    "Best_k_pcr": best_components_pcr,
    "non_zero_counts": nonzero_counts_en,
    "max_depths_rf" : max_depths_rf,
    "max_depths_gbrt" : max_depths_gbrt,
    "max_depths_xgb" : max_depths_xgb,
})


feature_importance_xgb_counts = [np.sum(imp > 0) for imp in feature_importance_xgb]

df_best_param["feature_importance_xgb"] = feature_importance_xgb_counts

# Puis dans ton plot
axs[5].plot(df_best_param["Date"], df_best_param["feature_importance_xgb"], color="darkcyan", marker='o')
axs[5].set_title("Number of features – XGBoost")


fig, axs = plt.subplots(3, 2, figsize=(12, 10), sharex=True)
axs = axs.flatten()

# PLS
axs[0].plot(df_best_param["Date"], df_best_param["Best_k_pls"], color="royalblue")
axs[0].set_title("Best number of components (PLS)")
axs[0].set_ylabel("k")
axs[0].grid(True)

# PCR
axs[1].plot(df_best_param["Date"], df_best_param["Best_k_pcr"], color="purple")
axs[1].set_title("Best number of components (PCR)")
axs[1].set_ylabel("k")
axs[1].grid(True)

# ElasticNet non-zero
axs[2].plot(df_best_param["Date"], df_best_param["non_zero_counts"], color="seagreen", marker='o')
axs[2].set_title("Model complexity – ElasticNet")
axs[2].set_ylabel("Non-zero counts")
axs[2].grid(True)

# RF max_depth
axs[3].plot(df_best_param["Date"], df_best_param["max_depths_rf"], color="firebrick", marker='o')
axs[3].set_title("Max Depth – Random Forest")
axs[3].set_ylabel("Max Depth")
axs[3].grid(True)

# GBRT complexity
axs[4].plot(df_best_param["Date"], complexity_gbrt, color="royalblue", marker='o')
axs[4].set_title("GBRT complexity (features with importance > 0)")
axs[4].set_ylabel("Number of features")
axs[4].grid(True)

# XGB max_depth
axs[5].plot(df_best_param["Date"], df_best_param["feature_importance_xgb"], color="darkcyan", marker='o')
axs[5].set_title("Number of features – XGBoost")
axs[5].set_ylabel("Number of features")
axs[5].grid(True)

for ax in axs:
    ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()


CALCULS / GRAPH → importance des variables

In [ ]:
import numpy as np
from sklearn.metrics import r2_score
import pandas as pd

def compute_variable_importance_gu(model, X_val, y_val, covariates):
    y_pred_baseline = model.predict(X_val)
    r2_baseline = r2(y_val, y_pred_baseline)
    importances = []
    for var in covariates:
        X_perturbed = X_val.copy()
        X_perturbed[var] = 0
        y_pred_perturbed = model.predict(X_perturbed)
        r2_perturbed = r2(y_val, y_pred_perturbed)
        loss = max(0, r2_baseline - r2_perturbed)
        importances.append(loss)
    return pd.DataFrame({'Variable': covariates, 'Importance': importances})


In [ ]:
#OLS
all_importances_ols = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    # Refit OLS
    ols = LinearRegression()
    ols.fit(x_trainval, y_trainval)

    # Importance Gu-style
    imp_df = compute_variable_importance_gu(ols, x_val[covariates], y_val, covariates)
    all_importances_ols.append(imp_df['Importance'].values)
# Moyenne sur les splits
mean_imp_ols = np.mean(all_importances_ols, axis=0)
importance_final_ols = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_ols
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

In [ ]:
#PLS
all_importances_pls = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_k = best_components_pls[split_idx]  # récupère le k optimal pour ce split

    # Refit PLS sur train+val
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    pls_model = PLSRegression(n_components=best_k, scale=False)
    pls_model.fit(x_trainval, y_trainval)

    # Calcul importance façon Gu
    imp_df = compute_variable_importance_gu(pls_model, x_val[covariates], y_val, covariates)
    all_importances_pls.append(imp_df['Importance'].values)

# Moyenne sur les splits
mean_imp_pls = np.mean(all_importances_pls, axis=0)
importance_final_pls = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_pls
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

In [ ]:
#PCR
all_importances_pcr = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_k = best_components_pcr[split_idx]

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    # Refit PCR pipeline sur train+val
    pcr_model = Pipeline([
        ('pca', PCA(n_components=best_k)),
        ('reg', LinearRegression())
    ])
    pcr_model.fit(x_trainval, y_trainval)

    # Importance façon Gu
    imp_df = compute_variable_importance_gu(pcr_model, x_val[covariates], y_val, covariates)
    all_importances_pcr.append(imp_df['Importance'].values)

# Moyenne des importances
mean_imp_pcr = np.mean(all_importances_pcr, axis=0)

importance_final_pcr = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_pcr
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


In [ ]:
#Elastic net
all_importances_en = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_lambda = best_lambdas[split_idx]

    # Refit sur train + val avec le bon lambda
    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    en_model = ElasticNet(alpha=best_lambda, l1_ratio=0.5, max_iter=10000)
    en_model.fit(x_trainval, y_trainval)

    # Importance selon Gu
    imp_df = compute_variable_importance_gu(en_model, x_val[covariates], y_val, covariates)
    all_importances_en.append(imp_df['Importance'].values)

# Moyenne sur les splits
mean_imp_en = np.mean(all_importances_en, axis=0)

importance_final_en = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_en
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


In [ ]:
# Random Forest 
all_importances_rf = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_params = best_params_rf[split_idx]

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    rf_model = RandomForestRegressor(
        **best_params,
        n_jobs=-1,
        random_state=0
    )
    rf_model.fit(x_trainval, y_trainval)

    # Importance Gu
    imp_df = compute_variable_importance_gu(rf_model, x_val[covariates], y_val, covariates)
    all_importances_rf.append(imp_df['Importance'].values)

# Moyenne sur les splits
mean_imp_rf = np.mean(all_importances_rf, axis=0)

importance_final_rf = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_rf
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


In [ ]:
#GBRT
all_importances_gbrt = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_params = best_params_gbrt[split_idx]

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    gbrt_model = GradientBoostingRegressor(**best_params, random_state=0)
    gbrt_model.fit(x_trainval, y_trainval)

    # Importance Gu
    imp_df = compute_variable_importance_gu(gbrt_model, x_val[covariates], y_val, covariates)
    all_importances_gbrt.append(imp_df['Importance'].values)

# Moyenne sur les splits
mean_imp_gbrt = np.mean(all_importances_gbrt, axis=0)

importance_final_gbrt = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_gbrt
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


In [ ]:
all_importances_xgb = []

for split_idx, (x_train, y_train, x_val, y_val, x_test, y_test) in enumerate(preprocessed_splits):
    best_params = best_params_xgb[split_idx]

    x_trainval = pd.concat([x_train[covariates], x_val[covariates]])
    y_trainval = pd.concat([y_train, y_val])

    model = XGBRegressor(**best_params, random_state=0, n_jobs=-1)
    model.fit(x_trainval, y_trainval)

    # Importance Gu
    imp_df = compute_variable_importance_gu(model, x_val[covariates], y_val, covariates)
    all_importances_xgb.append(imp_df['Importance'].values)

# Moyenne sur les splits
mean_imp_xgb = np.mean(all_importances_xgb, axis=0)

importance_final_xgb = pd.DataFrame({
    'Variable': covariates,
    'Importance': mean_imp_xgb
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)


In [ ]:
importance_final_ols["Model"] = "OLS"
importance_final_pls["Model"] = "PLS"
importance_final_pcr["Model"] = "PCR"
importance_final_en["Model"] = "EN"
importance_final_rf["Model"] = "RF"
importance_final_gbrt["Model"] = "GBRT"
importance_final_xgb["Model"] = "XGB"


importance_all = pd.concat([
    importance_final_ols,
    importance_final_pls,
    importance_final_pcr,
    importance_final_en,
    importance_final_rf,
    importance_final_gbrt,
    importance_final_xgb
], ignore_index=True)

import matplotlib.pyplot as plt

# Normalisation par modèle
importance_all["Importance"] = importance_all.groupby("Model")["Importance"].transform(lambda x: x / x.sum())

# Top 15 par modèle
top15_per_model = (
    importance_all.sort_values(by=["Model", "Importance"], ascending=[True, False])
    .groupby("Model")
    .head(15)
)

# Grille de subplots
models = top15_per_model["Model"].unique()
n_models = len(models)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), constrained_layout=True)

for ax, model_name in zip(axes.flat, models):
    df_model = top15_per_model[top15_per_model["Model"] == model_name]
    ax.barh(df_model["Variable"], df_model["Importance"])
    ax.set_title(model_name)
    ax.invert_yaxis()

# Supprimer les axes en trop
for i in range(len(models), n_rows * n_cols):
    fig.delaxes(axes.flat[i])

plt.suptitle("Top-15 Variable Importances by Model (Figure 4 style)", fontsize=16)
plt.show()


In [ ]:
import seaborn as sns

#top 20 → on met tout sur un df pour facilier les opérations : on normalise, puis trie par 20 
#ajoute une colonne model pur chaque modèle 
importance_final_ols["Model"] = "OLS"
importance_final_pls["Model"] = "PLS"
importance_final_pcr["Model"] = "PCR"
importance_final_en["Model"] = "EN"
importance_final_rf["Model"] = "RF"
importance_final_gbrt["Model"] = "GBRT"
importance_final_xgb["Model"] = "XGB"

#concatène en un df
importance_all = pd.concat([
    importance_final_ols,
    importance_final_pls,
    importance_final_pcr,
    importance_final_en,
    importance_final_rf,
    importance_final_gbrt,
    importance_final_xgb
], ignore_index=True)


# Étape 1 : normalisation par modèle
importance_all["Importance"] = importance_all.groupby("Model")["Importance"].transform(lambda x: x / x.sum())

# Étape 2 : top 20 par modèle
top20_all = (
    importance_all.sort_values(by=["Model", "Importance"], ascending=[True, False])
    .groupby("Model")
    .head(35)
)

# Étape 3 : pivot pour heatmap
df_plot = top20_all.pivot(index="Variable", columns="Model", values="Importance")

# (Optionnel) trier les lignes par importance moyenne
df_plot = df_plot.fillna(0)
df_plot = df_plot.loc[df_plot.mean(axis=1).sort_values(ascending=False).index]

# Étape 4 : Heatmap
plt.figure(figsize=(12, 12))
sns.heatmap(df_plot, cmap="YlGnBu", annot=True, fmt=".3f", linewidths=0.5)
plt.title("Variable Importances per Model (Figure 4 style)")
plt.tight_layout()
plt.show()


PORTEFEUILLES

In [ ]:
#I. PORTEFEUILLES EQUALLY WEIGHT
pred_cols = ['y_pred_ols','y_pred_pls','y_pred_pcr','y_pred_en',
             'y_pred_rf','y_pred_gbrt','y_pred_xgb','y_pred_ha']

col_ret = 'y_true'
nb_groups = 3  

# Résultats finaux
final_table = {}

for col_pred in pred_cols:
    all_rows = []

    for date, i in df_predict.groupby('Date'):
        i = i.sort_values(col_pred).reset_index(drop=True)
        n = len(i)
        size = n // nb_groups
        groups = [min(idx // size, nb_groups - 1) for idx in range(n)]
        i["group"] = groups

        for grp in range(nb_groups):
            sub = i[i["group"] == grp]
            mean_pred = sub[col_pred].mean()
            mean_real = sub[col_ret].mean()
            std_real = sub[col_ret].std()
            sr_real = mean_real / std_real if std_real != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    # Convertir en DataFrame
    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    # Ajouter H-L
    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  df_grouped.loc[nb_groups-1, "Avg"]  - df_grouped.loc[0, "Avg"],
        "SD":   df_grouped.loc[nb_groups-1, "SD"],  # ou recompute spread SD
        "SR":   (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table[col_pred.upper()] = df_grouped


for model, df in final_table.items():
    print(f"\n {model}")
    print(df.to_string(index=False))

In [ ]:
#on récupère market equity pour faire des me weighted portefeuilles
df_me = pd.read_excel("me_df.xlsx")

df_me = df_me[(df_me["Date"] >= "2007-12-01") & (df_me["Date"] <= "2020-11-30")]
df_me = df_me.drop(columns=["Unnamed: 0"])
df_me = df_me.sort_values(["Date", "Ticker"]).reset_index(drop=True)
df_me = df_me.rename(columns={"Mkt_Cap_Monthly": "mktcap"})
df_me["Date"] = pd.to_datetime(df_me["Date"]).dt.to_period("M")

df_portfolio_me = df_preds_oos.copy()
df_portfolio_me["Date"] = pd.to_datetime(df_preds_oos["Date"]).dt.to_period("M")


# Merge sur Date + Ticker
df_portfolio_me = df_portfolio_me.merge(df_me, on=["Date", "Ticker"], how="left")

In [ ]:

# 2. Portefeuilles market-weighted
pred_cols = ['y_pred_ols','y_pred_pls','y_pred_pcr','y_pred_en',
             'y_pred_rf','y_pred_gbrt','y_pred_xgb','y_pred_ha']

col_ret = 'y_true'
nb_groups = 3  
final_table_vw = {}

for col_pred in pred_cols:
    all_rows = []

    for date, group in df_portfolio_me.groupby('Date'):
        group = group.sort_values(col_pred).reset_index(drop=True)
        n = len(group)
        size = n // nb_groups
        group["group"] = [min(i // size, nb_groups - 1) for i in range(n)]

        for grp in range(nb_groups):
            sub = group[group["group"] == grp]
            if sub["mktcap"].sum() == 0:
                continue
            weights = sub["mktcap"] / sub["mktcap"].sum()
            mean_pred = (weights * sub[col_pred]).sum()
            mean_real = (weights * sub[col_ret]).sum()
            std_real = np.sqrt((weights * (sub[col_ret] - mean_real) ** 2).sum())
            sr_real = mean_real / std_real if std_real != 0 else np.nan

            all_rows.append({
                "Group": grp,
                "Pred": mean_pred,
                "Avg": mean_real,
                "SD": std_real,
                "SR": sr_real
            })

    df_result = pd.DataFrame(all_rows)
    df_grouped = df_result.groupby("Group").mean().reset_index()

    hl_row = {
        "Group": "H-L",
        "Pred": df_grouped.loc[nb_groups-1, "Pred"] - df_grouped.loc[0, "Pred"],
        "Avg":  df_grouped.loc[nb_groups-1, "Avg"]  - df_grouped.loc[0, "Avg"],
        "SD":   df_grouped.loc[nb_groups-1, "SD"],
        "SR":   (df_grouped.loc[nb_groups-1, "Avg"] - df_grouped.loc[0, "Avg"]) / df_grouped["SD"].mean()
    }

    df_grouped = pd.concat([df_grouped, pd.DataFrame([hl_row])], ignore_index=True)
    final_table_vw[col_pred.upper()] = df_grouped

# Affichage final des résultats market-weighted
for model, df in final_table_vw.items():
    print(f"\n{model} (Market Weighted)")
    print(df.to_string(index=False))


METRIQUES : R2 BENCHMARK : Analyse de pourquoi c'est mauvais

In [ ]:


print("OLS   : In-sample =", r2_in_ols,  "| OOS =", r2_oos_ols)
print("PLS   : In-sample =", r2_in_pls,  "| OOS =", r2_oos_pls)
print("PCR   : In-sample =", r2_in_pcr,  "| OOS =", r2_oos_pcr)
print("Enet  : In-sample =", r2_in_en,   "| OOS =", r2_oos_en)
print("Rf  : In-sample =", r2_in_rf,   "| OOS =", r2_oos_rf)
print("GBRT  : In-sample =", r2_in_gbrt,   "| OOS =", r2_oos_gbrt)
print("XGB : In-sample =", r2_in_xgb,   "| OOS =", r2_oos_xgb)
print("HA : In-sample =", r2_in_ha,   "| OOS =", r2_oos_ha)


In [ ]:
def compute_mse_diff(actual, y_benchmark, y_pred, k):
    return mean_squared_error(y_benchmark[:k], actual[:k]) - mean_squared_error(y_pred[:k], actual[:k])

df_oos = df_oos.sort_values(by="Date").reset_index(drop=True)

ml_diff_in_cumulative_mse = {}
for col in model_cols:  # ex: ['y_pred_ols', 'y_pred_pls', etc.]
    y_pred = df_oos[col].to_numpy()
    y_benchmark = df_oos["y_pred_ha"].to_numpy()
    y_true = df_oos["y_true"].to_numpy()
    
    diff_mse = [compute_mse_diff(y_true, y_benchmark, y_pred, k) for k in range(1, len(y_true) + 1)]
    ml_diff_in_cumulative_mse[col.replace("y_pred_", "")] = diff_mse

df_cumul_mse = pd.DataFrame(ml_diff_in_cumulative_mse)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 6))
for col in df_cum_mse.columns:
    plt.plot(df_cum_mse.index, df_cum_mse[col], label=col)

plt.xlabel("Date")
plt.ylabel("Cumulative MSE Gain vs HA")
plt.title("Cumulative MSE Difference: ML Models vs Historical Average")
plt.axhline(0, color="black", linestyle="--", linewidth=1)  # ligne horizontale à zéro
plt.legend()
plt.ylim(-0.005, 0.005)  # à adapter selon l’amplitude des faibles modèles
plt.grid(True)
plt.tight_layout()
plt.show()
